## Importación de librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


## Importación de datos

In [8]:
customers = pd.read_csv('../data/raw/customers.csv')
orders = pd.read_csv('../data/raw/orders.csv')
payments = pd.read_csv('../data/raw/order-payments.csv')
items = pd.read_csv('../data/raw/order-items.csv')
reviews = pd.read_csv('../data/raw/order-reviews.csv')

## Overview y chequeo general de los datos

In [ ]:
#breve overview del contenido de mis datos:
dataframes = { 
    'customers' : customers,
    'orders' : orders,
    'payments' : payments,
    'items' : items,
    'reviews' : reviews
}

for name, df in dataframes.items():
    profile = pd.DataFrame({
        'dtype' : df.dtypes,
        'cantidad nulos' : df.isnull().sum()
        })
    profile = profile.sort_values(by='cantidad nulos', ascending = False)

    print(f'\n====== {name.upper()} ======')
    print(profile)
    print('-' * 35)


====== CUSTOMERS ======
                          dtype  cantidad nulos
customer_id                 str               0
customer_unique_id          str               0
customer_zip_code_prefix  int64               0
customer_city               str               0
customer_state              str               0
-----------------------------------

====== ORDERS ======
                              dtype  cantidad nulos
order_delivered_customer_date   str            2965
order_delivered_carrier_date    str            1783
order_approved_at               str             160
order_id                        str               0
customer_id                     str               0
order_status                    str               0
order_purchase_timestamp        str               0
order_estimated_delivery_date   str               0
-----------------------------------

====== PAYMENTS ======
                        dtype  cantidad nulos
order_id                  str               0
payment_s

In [ ]:
#chequeo de estructura
print(f'Orders unicas: {orders['order_id'].nunique() == len(orders)}')

#chequeo reviews duplicadas
review_duplicadas = reviews.duplicated(subset='order_id').sum()

review_orders_afectadas = reviews[reviews.duplicated(subset='order_id', keep=False)]['order_id'].nunique()

print(f'Reviews duplicadas: {review_duplicadas}')
print(f'Orders afectadas: {review_orders_afectadas} ({review_orders_afectadas / len(reviews) *100:.2f}%)')

#chequeo nulos
print(f'Delivered date nulos: {orders['order_delivered_customer_date'].isnull().mean():.2%}')
print(f'Review score nulos: {reviews['review_score'].isnull().mean():.2%}')

Orders unicas: True
Reviews duplicadas: 559
Orders afectadas: 555 (0.56%)
Delivered date nulos: 2.98%
Review score nulos: 0.00%


## Limpieza y transformación de los datos

In [36]:
#chequeo y formateo de las variables 'fecha' para correcta operativa
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce'
)

In [37]:
#verifico orden cronológico de las columnas con formato 'fecha' y verifico que no haya entregas antes de realizada la compra
inconsistencias = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]

print(f"Registros con fechas ilógicas: {len(inconsistencias)}")


Registros con fechas ilógicas: 0


In [38]:
#chequeo el rango temporal de mi dataset
print(f"Inicio de datos: {orders['order_purchase_timestamp'].min()}")
print(f"Fin de datos: {orders['order_purchase_timestamp'].max()}")

Inicio de datos: 2016-09-04 21:15:19
Fin de datos: 2018-10-17 17:30:18


### Limpieza y transformación > Dataset Reviews

In [53]:
#confirmo reviews duplicadas
review_duplic = reviews.duplicated(subset='order_id', keep=False)
filas_duplic = reviews[review_duplic]

print(f'Filas reviews duplicadas: {len(filas_duplic)}')
print(f'Orders con reviews duplicadas: {filas_duplic['order_id'].nunique()}')
print("-" * 30)

Filas reviews duplicadas: 1114
Orders con reviews duplicadas: 555
------------------------------


In [54]:
#reviso qué hay dentro de las reviews duplicadas 
filas_duplic.groupby('order_id')['review_score'].nunique().value_counts()

review_score
1    346
2    209
Name: count, dtype: int64

In [55]:
#al constatar que son reviews de 1 y 2 estrellas (calificación mala) decido quedarme con la última review para caada order_id y así no sesgar mi variable review_score
reviews_sin_duplicados = reviews.drop_duplicates(subset='order_id', keep='last')

In [56]:
#verifico lo anterior
print(f'Reviews duplicadas: {reviews_sin_duplicados['order_id'].duplicated().sum()}')

Reviews duplicadas: 0


In [ ]:
#chequeo mi dataset de reviews 
profile_reviews = pd.DataFrame({
    'dtype': reviews.dtypes,
    'cantidad nulos': reviews.isnull().sum(),
    'porcentaje nulos': (reviews.isnull().sum() / len(reviews)) * 100
})

profile_reviews = profile_reviews.sort_values(by='cantidad nulos', ascending=False)

print(profile_reviews)

                                  dtype  cantidad nulos  porcentaje nulos
review_id                           str               0               0.0
order_id                            str               0               0.0
review_score                      int64               0               0.0
review_creation_date     datetime64[us]               0               0.0
review_answer_timestamp  datetime64[us]               0               0.0


### Como no abordaré en mi análisis el contenido en sí de las reviews (me limitaré a la calificación de las mismas) y dado la alta cantidad de nulos, decido eliminar las columnas 'review_comment_title' y 'review_comment_message'

In [ ]:
#elimino las columnas 'review_comment_title' y 'review_comment_message'
reviews = reviews.drop(columns=['review_comment_title', 'review_comment_message'])

KeyError: "['review_comment_title', 'review_comment_message'] not found in axis"

In [ ]:
#verifico lo anterior
print(reviews.columns)

Index(['review_id', 'order_id', 'review_score', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')


### Limpieza y transformación > Dataset Orders

In [ ]:
#chequeo las ordenes con status 'delivered' y nulos en 'order_delivered_customer_date' lo que sería un error de confirmación
nulo_error_entrega = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())
casos_error = orders[nulo_error_entrega]

#chequeo los nulos que sí son coherentes
nulo_normal = (orders['order_status'].isin(['shipped', 'canceled', 'unavailable', 'invoiced', 'processing'])) & \
                   (orders['order_delivered_customer_date'].isnull())
casos_normales = orders[nulo_normal]

print(f'--- Diagnóstico de Integridad ---')
print(f'Pedidos "delivered" sin fecha (Inconsistencias): {len(casos_error)}')
print(f'Pedidos en tránsito/cancelados sin fecha (Normal): {len(casos_normales)}')
print(f'Total de nulos en order_delivered_customer_date: {orders['order_delivered_customer_date'].isnull().sum()}')

--- Diagnóstico de Integridad ---
Pedidos "delivered" sin fecha (Inconsistencias): 8
Pedidos en tránsito/cancelados sin fecha (Normal): 2950
Total de nulos en order_delivered_customer_date: 2965

Muestra de registros inconsistentes:
                               order_id order_status order_purchase_timestamp
3002   2d1e2d5bf4dc7227b3bfebb81328c15f    delivered      2017-11-28 17:44:07
20618  f5dd62b788049ad9fc0526e3ad11a097    delivered      2018-06-20 06:58:43
43834  2ebdfc4f15f23b91474edf87475f108e    delivered      2018-07-01 17:05:11
79263  e69f75a717d64fc5ecdfae42b2e8e086    delivered      2018-07-01 22:05:55
82868  0d3268bad9b086af767785e3f0fc0133    delivered      2018-07-01 21:14:02


In [47]:
#procedo a eliminar los pedidos "delivered" sin fecha
total_inicial = len(orders)

condicion_error = (orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())

#aplico la limpieza utilizando el operador de negación (~), esto mantiene todo lo que no cumple la 'condicion_error'
orders = orders[~condicion_error].copy()

#verificación
total_final = len(orders)
eliminados = total_inicial - total_final

print(f"--- Resumen de Limpieza ---")
print(f"Registros inconsistentes eliminados: {eliminados}")
print(f"Registros restantes en 'orders': {total_final}")

--- Resumen de Limpieza ---
Registros inconsistentes eliminados: 8
Registros restantes en 'orders': 99433


In [48]:
#comprobación final para ver si quedan nulos con status 'delivered'
restantes_nulos_delivered = orders[(orders['order_status'] == 'delivered') & 
                                   (orders['order_delivered_customer_date'].isnull())]

print(f"Inconsistencias restantes: {len(restantes_nulos_delivered)}")

Inconsistencias restantes: 0


In [ ]:
#analizo la consistencia de los nulos 'order_approved_at' (160 nulos) y 'order_delivered_carrier_date' (1783 nulos)
nulos_approved = orders[orders['order_approved_at'].isnull()]['order_status'].value_counts()

nulos_carrier = orders[orders['order_delivered_carrier_date'].isnull()]['order_status'].value_counts()

print('--- Estado de pedidos con Fecha de Aprobación NULA ---')
print(nulos_approved)

print('\n--- Estado de pedidos con Fecha de Transportista NULA ---')
print(nulos_carrier)

--- Estados de pedidos con Fecha de Aprobación NULA ---
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

--- Estados de pedidos con Fecha de Transportista NULA ---
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        1
Name: count, dtype: int64


In [ ]:
#elimino los registros que son incosistentes como pedidos que figuran como 'entregado' pero la 'aprobación del pedido' es nula, o que la 'fecha de entrega al transportista' sea nula también.
inicial = len(orders)

#caso A: Entregados que no tienen fecha de aprobación
error_aprobacion = (orders['order_status'] == 'delivered') & (orders['order_approved_at'].isnull())

#caso B: Entregados que no tienen fecha de salida al transportista
error_carrier = (orders['order_status'] == 'delivered') & (orders['order_delivered_carrier_date'].isnull())

#aplico limpieza (eliminando ambos casos)
orders = orders[~(error_aprobacion | error_carrier)].copy()

print(f'Registros eliminados por inconsistencia logística: {inicial - len(orders)}')
print(f'Nuevos nulos en "order_approved_at": {orders['order_approved_at'].isnull().sum()}')
print(f'Nuevos nulos en "order_delivered_carrier_date": {orders['order_delivered_carrier_date'].isnull().sum()}')

Registros eliminados por inconsistencia logística: 15
Nuevos nulos en "order_approved_at": 146
Nuevos nulos en "order_delivered_carrier_date": 1781


#### Los nulos restantes son nulos 'coherentes' a la lógica operativa del e-commerce. Por ej.: un pedido cancelado no debería tener una 'order_approved', y un pedido en proceso o facturado tampoco debería necesariamente estar en manos del transportista. Eso ya corresponde a la eficiencia operativa que analizaré más adelante. 

### Antes de proceder al merge de mis datasets, hago una última verificación de nulos y formatos:

In [ ]:
 #hago una última comprobación de todos los datasets (previo al merge)
dataframes = { 
    'customers' : customers,
    'orders' : orders,
    'payments' : payments,
    'items' : items,
    'reviews' : reviews
}

for name, df in dataframes.items():
    profile = pd.DataFrame({
        'dtype' : df.dtypes,
        'cantidad nulos' : df.isnull().sum()
        })
    profile = profile.sort_values(by='cantidad nulos', ascending = False)

    print(f'\n====== {name.upper()} ======')
    print(profile)
    print('-' * 35)


====== CUSTOMERS ======
                          dtype  cantidad nulos
customer_id                 str               0
customer_unique_id          str               0
customer_zip_code_prefix  int64               0
customer_city               str               0
customer_state              str               0
-----------------------------------

====== ORDERS ======
                                        dtype  cantidad nulos
order_delivered_customer_date  datetime64[us]            2957
order_delivered_carrier_date   datetime64[us]            1781
order_approved_at              datetime64[us]             146
order_id                                  str               0
customer_id                               str               0
order_status                              str               0
order_purchase_timestamp       datetime64[us]               0
order_estimated_delivery_date  datetime64[us]               0
-----------------------------------

====== PAYMENTS ======
           